<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/File_Integrity_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python-based file integrity monitoring program that compares the trusted baseline SHA-256 hashes of important files with their current hashes and classifies each file as Unchanged, Modified, Newly Added, or Missing, while displaying both baseline and current hashes for modified files.

**Algorithm**

Create or load a trusted baseline containing file paths and SHA-256 hashes.

Select the directory containing the current files.

Calculate the SHA-256 hash of each current file.

Compare current file paths with the baseline.

If the file exists and hashes match, classify it as Unchanged.

If the file exists but hashes differ, classify it as Modified.

If a current file is not present in the baseline, classify it as Newly Added.

If a baseline file is not found in the current directory, classify it as Missing.

Display files requiring investigation.

Display baseline and current hashes for modified files.

In [1]:
# ==============================================
# FILE INTEGRITY MONITORING
# ==============================================

import os
import hashlib
import json
import pandas as pd

BASELINE_FILE = "/content/baseline.json"
MONITOR_DIR = "/content/important_files"

os.makedirs(MONITOR_DIR, exist_ok=True)

# ----------------------------------------------
# Function to Calculate SHA-256
# ----------------------------------------------

def calculate_sha256(filepath):

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as f:

        while True:

            data = f.read(4096)

            if not data:
                break

            sha256.update(data)

    return sha256.hexdigest()


# ----------------------------------------------
# Create Sample Files
# ----------------------------------------------

with open(MONITOR_DIR + "/config.txt", "w") as f:
    f.write("Original configuration")

with open(MONITOR_DIR + "/system.txt", "w") as f:
    f.write("Original system file")

with open(MONITOR_DIR + "/new_file.txt", "w") as f:
    f.write("New file")

# ----------------------------------------------
# Create Trusted Baseline
# ----------------------------------------------

baseline = {}

for filename in ["config.txt", "system.txt", "missing.txt"]:

    filepath = os.path.join(MONITOR_DIR, filename)

    if os.path.exists(filepath):

        baseline[filename] = calculate_sha256(filepath)

# Simulate trusted baseline for missing.txt
baseline["missing.txt"] = "a" * 64

with open(BASELINE_FILE, "w") as f:
    json.dump(baseline, f, indent=4)

print("=" * 75)
print("              FILE INTEGRITY MONITORING")
print("=" * 75)

print("\nTrusted baseline created.")

# ----------------------------------------------
# Simulate File Modification
# ----------------------------------------------

with open(MONITOR_DIR + "/config.txt", "a") as f:
    f.write("\nUnauthorized modification")

# ----------------------------------------------
# Read Baseline
# ----------------------------------------------

with open(BASELINE_FILE, "r") as f:
    baseline = json.load(f)

# ----------------------------------------------
# Scan Current Files
# ----------------------------------------------

current = {}

for root, dirs, files in os.walk(MONITOR_DIR):

    for filename in files:

        filepath = os.path.join(root, filename)

        if os.path.abspath(filepath) == os.path.abspath(BASELINE_FILE):
            continue

        relative_path = os.path.relpath(
            filepath, MONITOR_DIR
        )

        current[relative_path] = calculate_sha256(filepath)

# ----------------------------------------------
# Compare Baseline and Current
# ----------------------------------------------

results = []

all_files = set(baseline.keys()) | set(current.keys())

for filename in sorted(all_files):

    baseline_hash = baseline.get(filename)
    current_hash = current.get(filename)

    # Unchanged
    if baseline_hash and current_hash:

        if baseline_hash == current_hash:

            status = "UNCHANGED"

        else:

            status = "MODIFIED"

    # Missing
    elif baseline_hash and not current_hash:

        status = "MISSING"

    # Newly Added
    else:

        status = "NEWLY ADDED"

    results.append({
        "File": filename,
        "Status": status,
        "Baseline_SHA256": baseline_hash or "N/A",
        "Current_SHA256": current_hash or "N/A"
    })

# ----------------------------------------------
# Display Results
# ----------------------------------------------

report = pd.DataFrame(results)

print("\n" + "-" * 75)
print("INTEGRITY SCAN RESULTS")
print("-" * 75)

print(
    report[
        ["File", "Status"]
    ].to_string(index=False)
)

# ----------------------------------------------
# Investigation Report
# ----------------------------------------------

print("\n" + "=" * 75)
print("             FILES REQUIRING INVESTIGATION")
print("=" * 75)

investigation = report[
    report["Status"].isin(
        ["MODIFIED", "NEWLY ADDED", "MISSING"]
    )
]

if len(investigation) == 0:

    print("No files require investigation.")

else:

    for _, row in investigation.iterrows():

        print("\nFile   :", row["File"])
        print("Status :", row["Status"])

        if row["Status"] == "MODIFIED":

            print("Baseline SHA-256:")
            print(row["Baseline_SHA256"])

            print("Current SHA-256:")
            print(row["Current_SHA256"])

        elif row["Status"] == "NEWLY ADDED":

            print("Current SHA-256:")
            print(row["Current_SHA256"])

        elif row["Status"] == "MISSING":

            print("Baseline SHA-256:")
            print(row["Baseline_SHA256"])

# ----------------------------------------------
# Save Report
# ----------------------------------------------

report.to_csv(
    "/content/integrity_report.csv",
    index=False
)

print("\n" + "=" * 75)
print("Report saved as: /content/integrity_report.csv")
print("=" * 75)

              FILE INTEGRITY MONITORING

Trusted baseline created.

---------------------------------------------------------------------------
INTEGRITY SCAN RESULTS
---------------------------------------------------------------------------
        File      Status
  config.txt    MODIFIED
 missing.txt     MISSING
new_file.txt NEWLY ADDED
  system.txt   UNCHANGED

             FILES REQUIRING INVESTIGATION

File   : config.txt
Status : MODIFIED
Baseline SHA-256:
ae332444981af07ebc1e879e75f9b1a8ad309cd73daee5a212ece2d7d10067da
Current SHA-256:
ec0d27f4635cb6de01041d73a7886c14cfeb72cadcb1359f4274b43f493a968e

File   : missing.txt
Status : MISSING
Baseline SHA-256:
aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

File   : new_file.txt
Status : NEWLY ADDED
Current SHA-256:
44dedd0f97fea3c80ee8c85de1b67b6422208445fa83c5dd060e1b1ab21ea4da

Report saved as: /content/integrity_report.csv


**Result**

The Python file integrity monitoring program successfully compared the trusted baseline SHA-256 values with the hashes of current files. It correctly classified files as Unchanged, Modified, Newly Added, or Missing. For modified files, both the baseline and current SHA-256 values were displayed, allowing investigators to verify the detected change. Files classified as Modified, Newly Added, or Missing were highlighted as requiring further investigation.